# Exploration de CamemBERT : représenter des définitions par des vecteurs

> **Origine** : adapté d'un tutoriel conçu dans le cadre des journées *Deep Voice* de l'IRCAM par Roman Castagné, Nathan Godey et Benjamin Muller, dont il reprend les fonctions de visualisation et de calcul des représentations de phrases. Les cellules ont été modifiées et réduites pour ce projet.

[CamemBERT](https://camembert-model.fr/) est un modèle de langue pour le français (architecture RoBERTa). Ce notebook explore le modèle et teste trois façons de mesurer la **similarité entre deux définitions** ; la deuxième (moyenne des embeddings) est celle retenue dans `03_pipeline_graphe`.

In [ ]:
from pprint import pprint

import numpy as np
import torch
import torch.nn.functional as F
import plotly.express as px
from transformers import CamembertForMaskedLM, AutoTokenizer

## 1. Le modèle

In [ ]:
camembert = CamembertForMaskedLM.from_pretrained('camembert-base')

In [ ]:
# Visualiser la couches d'embeddings
camembert.roberta.embeddings

In [ ]:
# Visualiser les couches cachées
print(f"Camembert est composé de {len(camembert.roberta.encoder.layer)} couches cachées\n")
print("Voilà une couche cachée :")
camembert.roberta.encoder.layer[0]

Dans la plupart des couches du modèle, les vecteurs utilisés sont de dimension 768. C'est ce qu'on appelle la *dimension cachée* du modèle.

On peut aussi calculer le poids total du modèle en Mo :

In [ ]:
def display_nb_params(model):
    param_size = 0
    param_count = 0
    for param in model.parameters():
        param_count += param.nelement()
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    size_all_mb = (param_size + buffer_size) / 1024**2
    print('Nombre de paramètres: {} | Poids du modèle: {:.2f}MB'.format(param_count, size_all_mb))

display_nb_params(camembert)

## 2. Tokenisation de définitions

On utilise des définitions de dictionnaire dont le mot défini est remplacé par `<mask>`.

In [ ]:
batch_sentences = [
"<mask>: nom féminin. Fruit comestible de l'oranger, agrume d'un jaune tirant sur le rouge",
"<mask>: adjectif. Intermédiaire entre le bleu et le jaune ; qui a la couleur dominante de la végétation",
"<mask>: adjectif. De la couleur de l'orange",
"<mask>: adjectif. Qui est de la couleur du sang, du rubis, etc. (extrémité du spectre solaire)",
"<mask>: nom féminin. Fruit du pommier, rond, à pulpe ferme et juteuse",
"<mask>: nom féminin. Ensemble des sciences qui ont pour objet la quantité et l'ordre",
]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('camembert-base')

tokenizer_output = tokenizer(
    batch_sentences,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)
pprint(tokenizer_output, width=150)

Le pré-traitement est terminé : on peut donner le contenu du dictionnaire issu du tokenizer en entrée du modèle.

In [ ]:
with torch.no_grad():
    model_output = camembert(**tokenizer_output, output_hidden_states=True)

## 3. Prédiction du mot masqué

Pour chaque définition, les 10 mots que CamemBERT juge les plus probables à la place de `<mask>`.

In [ ]:
def get_probas_from_logits(logits):
    return logits.softmax(-1)


def visualize_mlm_predictions(tokenizer_output, model_output, tokenizer, nb_candidates=10):
    # Decode the tokenized sentences and clean-up the special tokens
    decoded_tokenized_sents = [sent.replace('<pad>', '').replace('<mask>', ' <mask>') for sent in tokenizer.batch_decode(tokenizer_output.input_ids)]

    # Retrieve the probas at the masked positions
    masked_tokens_mask = tokenizer_output.input_ids == tokenizer.mask_token_id
    batch_mask_probas = get_probas_from_logits(model_output.logits[masked_tokens_mask])

    for sentence, mask_probas in zip(decoded_tokenized_sents, batch_mask_probas):
        # Get top probas and plot them
        top_probas, top_token_ids = mask_probas.topk(nb_candidates, -1)
        top_tokens = tokenizer.convert_ids_to_tokens(top_token_ids)
        bar_chart = px.bar({"tokens": top_tokens[::-1], "probas": list(top_probas)[::-1]},
                        x="probas", y="tokens", orientation='h', title=sentence, width=800)
        bar_chart.show(config={'staticPlot': True})

visualize_mlm_predictions(tokenizer_output, model_output, tokenizer)

## 4. Représentation d'une phrase

Les `hidden_states` de la dernière couche donnent un vecteur par token (ici 512 positions après *padding*). Pour représenter toute la phrase on peut :

- prendre le vecteur du **premier token** ;
- faire la **moyenne** des vecteurs des tokens.

In [ ]:
token_embeddings = model_output.hidden_states[-1]

In [ ]:
def take_first_embedding(embeddings, attention_mask=None):
    return embeddings[:, 0]

def average_embeddings(embeddings, attention_mask):
    return (attention_mask[..., None] * embeddings).mean(1)

In [ ]:
first_tok_sentence_representations = take_first_embedding(token_embeddings, tokenizer_output.attention_mask)
avg_sentence_representations = average_embeddings(token_embeddings, tokenizer_output.attention_mask)
# average_sentence_representations = average_sentence(token_embeddings, tokenizer_output.attention_mask)

first_tok_sentence_representations.shape, avg_sentence_representations.shape

## 5. Similarité entre définitions

Similarité cosinus entre toutes les paires de définitions, selon les trois représentations (premier token, moyenne, moyenne position par position).

In [ ]:
for i in range(len(batch_sentences)):
    for j in range(len(batch_sentences)):
        first_tok_similarity_score = F.cosine_similarity(first_tok_sentence_representations[i], first_tok_sentence_representations[j], dim=-1)
        avg_similarity_score = F.cosine_similarity(avg_sentence_representations[i], avg_sentence_representations[j], dim=-1)
        # moyenne, sur les 512 positions, de la similarité entre les embeddings de chaque token
        token_scores = F.cosine_similarity(token_embeddings[i], token_embeddings[j], dim=-1)
        avg_token_score = token_scores.mean()

        print(f"{batch_sentences[i]}    vs.    {batch_sentences[j]}")
        print(f"Score (first_tok) : {first_tok_similarity_score}")
        print(f"Score (average) : {avg_similarity_score}")
        print(f"Score (average embedding) : {avg_token_score}\n")